# Are we solving the same problem?

The handoff describes the problem in words. Two people can read the same
description and end up with instances that differ slightly — a capacity derived
from $Q_0$ instead of $Q_d$, distances left unrounded — and then the two sets of
results are not about the same problem and there is nothing useful to compare.

This notebook checks that question against **your own code**. It contains no
implementation of the problem — only the numbers your implementation should
produce. You wire your functions into the four adapters in section 1, run the
notebook, and see whether the numbers come out.

If everything passes, we are solving the same problem, and your fronts can be
compared with the reference numbers in the handoff.

Nothing here touches your algorithm. These checks are about the instance and
the objectives only — how your search explores them is entirely your business.

## 1. Wire in your code

Fill in the four adapters below so they call into your implementation. They are
deliberately thin: if your internals are shaped differently, do the translation
here rather than changing your solver.

You may find that one of these is awkward to write — for instance if your code
can only evaluate whole solutions and not a single route. That is worth noting
in itself; it usually means the objective is entangled with the search, which
makes it harder to test.

In [6]:
import math
from pathlib import Path

import numpy as np

DATA_DIR = Path("data/CVRP")
INSTANCES = ["X-n101-k25", "X-n106-k14", "X-n110-k13", "X-n115-k10"]

# importing my code
import sys
sys.path.append(str(Path.cwd().parent))
from gvrp.emissions import *
from pyvrp import read, Model

RHO = GAMMA = 2.61
F0 = 0.30
ALPHA = 0.20
BETA = 1.0         
MULT = {"diesel": 1.00, "clean": 0.45}
L_NOM = 0.50
CO2_SCALE = 100

# --- adapter 1 -----------------------------------------------------------
# Parse a .vrp file with YOUR parser and return a dict with these keys
# (translate from whatever your code uses internally):
#     dist   : (n, n) array of distances, nodes in file order
#     demand : (n,)   array of demands, 0 at the depot
#     depot  : int    index of the depot
#     Q0     : int    base capacity from the CAPACITY header
#     K0     : int    base fleet size used in this study
def my_parse(path):
    data = read(str(path), round_func="round")
    dist = np.asarray(data.distance_matrix(0)).astype(np.int64)
    n = dist.shape[0]
    demand = np.zeros(n, dtype=np.int64)
    for i, client in enumerate(data.clients(), start=1):
        d = client.delivery
        demand[i] = int(d[0]) if isinstance(d, (list, tuple, np.ndarray)) else int(d)
 
    Q0 = int(Model.from_data(data).vehicle_types[0].capacity[0])
    K0 = len(data.clients())          # K0 = n
 
    return dict(dist=dist, demand=demand, depot=0, Q0=Q0, K0=K0)


# --- adapter 2 -----------------------------------------------------------
# Derive the heterogeneous fleet with YOUR code.
# Return dict(Kd=..., Kc=..., Qd=..., Qc=...).
def my_derive_fleet(K0, Q0):
    Kc = K0 // 2
    Kd = K0 - Kc
    import gvrp.config as C
    caps = C.derived_capacities(Q0)
    Qd = caps["diesel"]
    Qc = caps["clean"]

    return dict(Kd=Kd, Kc=Kc, Qd=Qd, Qc=Qc)


# --- adapter 3 -----------------------------------------------------------
# CO2 of ONE route with YOUR emission code.
#     route    : list of customer indices, depot excluded, in visiting order
#     inst     : dict as returned by my_parse
#     capacity : capacity of the vehicle serving this route
#     mu       : emission multiplier of that vehicle type

# shim: lets obj2.true_co2 run on a plain dict instance
class _Client:
    __slots__ = ("delivery",)
    def __init__(self, q):
        self.delivery = [int(q)]
 
 
class _DataShim:
    """Combines the two things obj2.true_co2 touches on a ProblemData."""
    def __init__(self, inst):
        self._clients = [_Client(q) for q in inst["demand"][1:]]
        self._dist = np.asarray(inst["dist"])
 
    def clients(self):
        return self._clients
 
    def distance_matrix(self, profile):
        return self._dist

def my_route_emission(route, inst, capacity, mu):
    label = "diesel" if abs(mu - 1.00) < 1e-12 else "clean"
    from gvrp.emissions import true_co2
    return true_co2(
        [(list(route), 0)],                  # assignments form
        _DataShim(inst),
        {0: label},                          # type_labels
        {label: int(capacity)},              # Q_by_type
        F0, ALPHA, BETA, GAMMA, {label: mu},
    )


# --- adapter 4 -----------------------------------------------------------
# Score a whole solution with YOUR objective code. Return (f1, f2).
#     routes : list of routes (lists of customer indices, depot excluded)
#     types  : list of "diesel"/"clean", one per route
def my_evaluate(routes, types, inst, fleet):
    D = np.asarray(inst["dist"])
 
    f1 = 0.0
    for r in routes:
        seq = [inst["depot"]] + list(r) + [inst["depot"]]
        f1 += sum(float(D[a][b]) for a, b in zip(seq[:-1], seq[1:]))
 
    type_labels = {0: "diesel", 1: "clean"}
    Q_by_type = {"diesel": fleet["Qd"], "clean": fleet["Qc"]}
    assignments = [(list(r), 0 if t == "diesel" else 1) for r, t in zip(routes, types)]

    from gvrp.emissions import true_co2
    f2 = true_co2(
        assignments, _DataShim(inst), type_labels, Q_by_type,
        F0, ALPHA, BETA, GAMMA, MULT,
    )
    return float(f1), float(f2)

## 2. Two conventions worth stating

Both are easy to get wrong and neither produces an error.

**Distances are rounded to the nearest integer** (CVRPLIB `EUC_2D`). Raw
floating-point Euclidean distances make every $f_1$ differ by a fraction of a
percent — small enough to look plausible, large enough that best-distance values
cannot be compared.

**$K_0 = n$, the number of customers** — not the `k` in the filename.
`X-n101-k25` has 100 customers, so $K_0 = 100$, not 25. The `k25` belongs to the
original homogeneous instance and is not used anywhere in this study.

In [7]:
missing = [n for n in INSTANCES if not (DATA_DIR / f"{n}.vrp").exists()]
assert not missing, f"Missing .vrp files: {missing} (check DATA_DIR)"

instances = {}
for name in INSTANCES:
    inst = my_parse(DATA_DIR / f"{name}.vrp")
    instances[name] = (inst, my_derive_fleet(inst["K0"], inst["Q0"]))

print("parsed all 4 instances")

parsed all 4 instances


## 3. Check 1 — the derived fleet

$$K_c = \lfloor K_0/2 \rfloor, \qquad K_d = K_0 - K_c$$
$$Q_d = \operatorname{round}(1.10\,Q_0), \qquad Q_c = \operatorname{round}(0.80\,Q_d)$$

Note $Q_c$ comes from $Q_d$, **not** from $Q_0$. Deriving it from $Q_0$ gives a
different number on every instance, which is the most common way this goes
wrong.

In [8]:
EXPECTED_FLEET = {
    #             K0    Q0   Kd   Kc   Qd   Qc
    "X-n101-k25": (100, 206, 50, 50, 227, 182),
    "X-n106-k14": (105, 600, 53, 52, 660, 528),
    "X-n110-k13": (109,  66, 55, 54,  73,  58),
    "X-n115-k10": (114, 169, 57, 57, 186, 149),
}

print(f"{'instance':<14}{'K0':>5}{'Q0':>6}{'Kd':>5}{'Kc':>5}{'Qd':>6}{'Qc':>6}")
for name in INSTANCES:
    inst, fleet = instances[name]
    got = (inst["K0"], inst["Q0"], fleet["Kd"], fleet["Kc"], fleet["Qd"], fleet["Qc"])
    print(f"{name:<14}" + "".join(f"{v:>{w}}" for v, w in zip(got, (5, 6, 5, 5, 6, 6))))
    assert got == EXPECTED_FLEET[name], f"{name}: got {got}, expected {EXPECTED_FLEET[name]}"

print("\nOK - derived fleet matches the handoff for all 4 instances.")

instance         K0    Q0   Kd   Kc    Qd    Qc
X-n101-k25      100   206   50   50   227   182
X-n106-k14      105   600   53   52   660   528
X-n110-k13      109    66   55   54    73    58
X-n115-k10      114   169   57   57   186   149

OK - derived fleet matches the handoff for all 4 instances.


## 4. Check 2 — the parsed instances

Check 1 only looked at the capacity header and the node count. These totals go
over the parsed arrays themselves.

The one that matters most is the distance sum: if distances are not rounded to
the nearest integer, this is the check that will say so.

In [9]:
EXPECTED_PARSE = {
    "X-n101-k25": dict(n_nodes=101, depot=0, demand_sum=5147, dist_sum=5172658, dist_max=1265),
    "X-n106-k14": dict(n_nodes=106, depot=0, demand_sum=7864, dist_sum=3934714, dist_max=1318),
    "X-n110-k13": dict(n_nodes=110, depot=0, demand_sum= 816, dist_sum=6136444, dist_max=1278),
    "X-n115-k10": dict(n_nodes=115, depot=0, demand_sum=1535, dist_sum=6543654, dist_max=1262),
}

print(f"{'instance':<14}{'nodes':>7}{'depot':>7}{'demand sum':>12}{'dist sum':>12}{'dist max':>10}")
for name in INSTANCES:
    inst, _ = instances[name]
    got = dict(
        n_nodes=len(inst["demand"]),
        depot=int(inst["depot"]),
        demand_sum=int(np.sum(inst["demand"])),
        dist_sum=int(np.sum(inst["dist"])),
        dist_max=int(np.max(inst["dist"])),
    )
    print(f"{name:<14}{got['n_nodes']:>7}{got['depot']:>7}{got['demand_sum']:>12}"
          f"{got['dist_sum']:>12}{got['dist_max']:>10}")
    assert got == EXPECTED_PARSE[name], (name, got, EXPECTED_PARSE[name])

print("\nOK - parsed instances match on all 4.")

instance        nodes  depot  demand sum    dist sum  dist max
X-n101-k25        101      0        5147     5172658      1265
X-n106-k14        106      0        7864     3934714      1318
X-n110-k13        110      0         816     6136444      1278
X-n115-k10        115      0        1535     6543654      1262

OK - parsed instances match on all 4.


## 5. Check 3 — the emission formula

$$e(d, \lambda, \mu_v) = \rho\,F_0\,(1 + \alpha\lambda)\,\mu_v\,d$$

with $\rho = 2.61$, $F_0 = 0.30$, $\alpha = 0.20$, and $\mu_d = 1.00$,
$\mu_c = 0.45$. The load bookkeeping is the part that is easy to get wrong:

- the vehicle leaves the depot carrying the **whole route demand**;
- $\lambda$ for the arc *into* customer $i$ uses the load **before** serving $i$;
- the demand is subtracted on arrival, so the return arc is travelled empty
  ($\lambda = 0$);
- $\lambda$ is clipped to $[0,1]$.

So a route's emissions depend on the **order** of its customers, not just the
set. Serving heavy customers early lowers the load carried over the remaining
arcs — that is what makes $f_2$ genuinely different from $f_1$ rather than a
rescaling of it.

The instance below is small enough to check by hand. Depot at the origin, 4
customers, $Q_0 = 50$ and $K_0 = 4$, hence $Q_d = 55$, $Q_c = 44$,
$K_d = K_c = 2$.

| node | coords | demand |
|---|---|---|
| 0 (depot) | (0, 0) | – |
| 1 | (3, 4) | 10 |
| 2 | (6, 8) | 20 |
| 3 | (0, 10) | 25 |
| 4 | (0, 20) | 25 |

Expected arc by arc, for route $A = [1,2]$ on a clean vehicle and $B = [3,4]$
on a diesel one:

| route | arc | $d$ | $\lambda$ | factor | arc CO₂ |
|---|---|---|---|---|---|
| A (clean, $Q_c$=44, $\mu$=0.45) | 0→1 | 5 | 0.681818 | 0.40039773 | 2.001989 |
| | 1→2 | 5 | 0.454545 | 0.38438182 | 1.921909 |
| | 2→0 | 10 | 0.000000 | 0.35235000 | 3.523500 |
| | **total** | | | | **7.44739773** |
| B (diesel, $Q_d$=55, $\mu$=1.00) | 0→3 | 10 | 0.909091 | 0.92536364 | 9.253636 |
| | 3→4 | 10 | 0.454545 | 0.85418182 | 8.541818 |
| | 4→0 | 20 | 0.000000 | 0.78300000 | 15.660000 |
| | **total** | | | | **33.45545455** |

In [10]:
toy_coords = {0: (0.0, 0.0), 1: (3.0, 4.0), 2: (6.0, 8.0), 3: (0.0, 10.0), 4: (0.0, 20.0)}
toy_demand = np.array([0, 10, 20, 25, 25])

n = len(toy_coords)
toy_dist = np.zeros((n, n))
for a in range(n):
    for b in range(n):
        toy_dist[a, b] = round(math.hypot(toy_coords[a][0] - toy_coords[b][0],
                                          toy_coords[a][1] - toy_coords[b][1]))

toy = dict(dist=toy_dist, demand=toy_demand, depot=0, Q0=50, K0=4)
toy_fleet = my_derive_fleet(toy["K0"], toy["Q0"])
print("your toy fleet:", toy_fleet)
assert toy_fleet["Qd"] == 55 and toy_fleet["Qc"] == 44
assert toy_fleet["Kd"] == 2 and toy_fleet["Kc"] == 2

e_A = my_route_emission([1, 2], toy, toy_fleet["Qc"], 0.45)
e_B = my_route_emission([3, 4], toy, toy_fleet["Qd"], 1.00)
print(f"route A (clean)  = {e_A:.8f}   expected 7.44739773")
print(f"route B (diesel) = {e_B:.8f}   expected 33.45545455")

assert abs(e_A - 7.44739773) < 1e-6, e_A
assert abs(e_B - 33.45545455) < 1e-6, e_B

print("\nOK - emission formula matches.")

your toy fleet: {'Kd': 2, 'Kc': 2, 'Qd': 55, 'Qc': 44}
route A (clean)  = 7.44739773   expected 7.44739773
route B (diesel) = 33.45545455   expected 33.45545455

OK - emission formula matches.


## 6. Check 4 — scoring a whole solution

A solution is a pair `(routes, types)`: `routes[i]` is a list of customer
indices with the depot excluded, and `types[i]` is `"diesel"` or `"clean"` for
that route.

The vehicle type of each route is a **decision variable** — your algorithm
chooses it, alongside the routing. A solution is feasible only if every customer
is visited once, each route's load fits **its own** assigned vehicle ($Q_c$ for
clean, $Q_d$ for diesel), and there are at most $K_d$ diesel and $K_c$ clean
routes.

Because clean vehicles are both smaller ($Q_c < Q_d$) and cleaner
($\mu_c = 0.45$), the assignment is itself a trade-off: fewer, fuller routes give
a lower $f_1$, but they are too heavy for clean vehicles and so give a higher
$f_2$. Much of the shape of the front comes from this.

On the toy instance, $A = [1,2]$ clean and $B = [3,4]$ diesel is the only
feasible assignment, since $B$'s load of 50 exceeds $Q_c = 44$.

In [11]:
f1, f2 = my_evaluate([[1, 2], [3, 4]], ["clean", "diesel"], toy, toy_fleet)
print(f"f1 = {f1:.6f}   expected 60")
print(f"f2 = {f2:.6f}   expected 40.902852")

assert abs(f1 - 60.0) < 1e-6, f1
assert abs(f2 - 40.902852) < 1e-5, f2
assert abs(f2 - (e_A + e_B)) < 1e-6, "f2 should be the sum of the two route emissions"

print("\nOK - whole-solution scoring matches.")

f1 = 60.000000   expected 60
f2 = 40.902852   expected 40.902852

OK - whole-solution scoring matches.


## 7. Check 5 — one solution from your own solver

Run your solver briefly on `X-n110-k13` and return one feasible solution as
`(routes, types)`.

There is nothing to compare against here — the point is that a solution your
search actually produces goes through your own evaluator and comes back with
objective values consistent with the checks above. If your solver reports its
own $f_1$/$f_2$ internally, compare those too: they must agree.

In [12]:
import numpy as np
import gvrp.config as C
 
SOLVER_PATH = str(DATA_DIR / "X-n110-k13.vrp")
 
C.INSTANCE_PATH = SOLVER_PATH
C.BUDGET_SECONDS = 25          
C.POP_SIZE = 20               
C.K0 = C.customers_in(SOLVER_PATH)
C.N_CLEAN = C.K0 // 2
C.N_DIESEL = C.K0 - C.N_CLEAN

import sys
import gvrp.emissions
sys.modules["config"] = C
sys.modules["obj2"] = gvrp.emissions 
import gvrp.nsga2 as G
 
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pyvrp import read

def my_solution(inst, fleet):
    LS_BUDGET, GA_BUDGET, SEED = 20, "00:00:15", 1
 
    raw = read(SOLVER_PATH, round_func="round")
    base_cap = int(raw.vehicle_types()[0].capacity[0])
    Q_by_type = C.derived_capacities(base_cap)
 
    # The solver reads its instance from disk
    assert Q_by_type["diesel"] == fleet["Qd"], (Q_by_type, fleet)
    assert Q_by_type["clean"] == fleet["Qc"], (Q_by_type, fleet)
    assert int(np.sum(inst["demand"])) == sum(
        int(c.delivery[0]) for c in raw.clients()
    ), "notebook and solver are not on the same instance"
 
    dist_data, co2_data = G.build_problem_datas(
        raw, G.N_DIESEL, G.N_CLEAN, Q_by_type
    )
    dls = G.DualLocalSearch(
        dist_data, co2_data, Q_by_type, G.N_DIESEL, G.N_CLEAN,
        seed=SEED, nb_size=G.LS_NEIGHBOURHOOD_SIZE, time_budget=LS_BUDGET,
    )
    problem = G.GreenVRPProblem(dist_data, Q_by_type, G.N_DIESEL, G.N_CLEAN)
 
    algorithm = NSGA2(
        pop_size=C.POP_SIZE,
        sampling=G.VRPSampling(dls)._do(problem, C.POP_SIZE),
        crossover=G.VRPCrossover(dls, prob=G.CX_PROB),
        mutation=G.VRPMutation(dls, rate=G.MUT_RATE),
        eliminate_duplicates=G.VRPDuplicateElimination(),
    )
 
    res = minimize(problem, algorithm, ("time", GA_BUDGET), seed=SEED,
                   verbose=False, copy_algorithm=False)
 
    # Lowest-distance feasible individual in the final population.
    best = None
    for row in np.atleast_2d(res.X):
        m = G.evaluate_solution(row[0], dist_data, Q_by_type,
                                G.N_DIESEL, G.N_CLEAN)
        if m is not None and m[2] == 0.0 and (best is None or m[0] < best[1][0]):
            best = (row[0], m)
    assert best is not None, "solver returned no feasible solution"
 
    sol, metrics = best
    routes, types = [], []
    for visits, vt in G.route_list(sol):
        if visits:
            routes.append(list(visits))
            types.append(G.TYPE_LABELS[vt])
 
    # The solver's own objectives, for the cross-check at the bottom.
    my_solution.f1, my_solution.f2 = float(metrics[0]), float(metrics[1])
    return routes, types


inst, fleet = instances["X-n110-k13"]
routes, types = my_solution(inst, fleet)

# Feasibility, per the rules above.
visited = sorted(c for r in routes for c in r)
assert visited == sorted(i for i in range(len(inst["demand"])) if i != inst["depot"]), \
    "every customer must be visited exactly once"
assert len(routes) == len(types)
assert types.count("diesel") <= fleet["Kd"], "too many diesel routes"
assert types.count("clean") <= fleet["Kc"], "too many clean routes"
for r, t in zip(routes, types):
    cap = fleet["Qc"] if t == "clean" else fleet["Qd"]
    load = sum(int(inst["demand"][c]) for c in r)
    assert load <= cap, f"{t} route load {load} exceeds capacity {cap}"

f1, f2 = my_evaluate(routes, types, inst, fleet)
print(f"routes={len(routes)}  clean={types.count('clean')}  "
      f"diesel={types.count('diesel')}  f1={f1:.0f}  f2={f2:.4f}")

# If your solver computes its own objectives, they must agree:
# my_f1, my_f2 = ...
# assert abs(my_f1 - f1) < 1e-6, (my_f1, f1)
# assert abs(my_f2 - f2) < 1e-6, (my_f2, f2)

print("\nOK - solver output is feasible and scores consistently.")

routes=12  clean=1  diesel=11  f1=14369  f2=11655.8706

OK - solver output is feasible and scores consistently.


## 8. Sanity range once the study has run

The handoff's reference tables tell you whether the pipeline is in the right
units. Use the **best distance** and **best CO₂** columns for that:

| instance | best distance | best CO₂ |
|---|---|---|
| X-n101-k25 | ≈ 25 900 – 26 800 | ≈ 11 880 – 11 900 |
| X-n106-k14 | ≈ 23 900 – 24 950 | ≈ 11 460 – 11 600 |
| X-n110-k13 | ≈ 14 260 – 14 340 | ≈ 6 250 – 6 285 |

For a sense of scale, an unoptimised solution on `X-n110-k13` — customers in
index order, packed to capacity — scores around 60 000 on distance, against
roughly 14 300 once the search has run. So if finished runs still land near
60 000, the problem is in the pipeline rather than in the algorithm.

**HV, IGD and SP are results, not targets.** They depend on the algorithm and on
the normalisation bounds, and they are not expected to match. Don't tune towards
them.